In [ ]:
# import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
# Load dataset
df=pd.read_csv("Mall_Customers.csv")
df

In [ ]:
# shape
df.shape

In [ ]:
# columns info
df.info()

In [ ]:
# Check null values
df.isnull().sum()

In [ ]:
# Check duplicated
df.duplicated().sum()

In [ ]:
# Check data consinstence
df["Gender"].value_counts()

In [ ]:
#Check impossile values
for col in df.select_dtypes(include="number").columns:
    a=df[col]<0
    if a.sum()>0:
        print(df.loc[a])

MODEL

In [ ]:
data=(df.copy()).drop(columns="CustomerID")

In [ ]:
numerical_cols=data.select_dtypes(include="number").columns
numerical_pipeline=Pipeline([
    ("scaler",StandardScaler())
])

In [ ]:
categorical_cols=data.select_dtypes(include="str").columns
categorical_pipeline=Pipeline([
    ("encoder",OneHotEncoder(handle_unknown='ignore'))
])

In [ ]:
preprocessor=ColumnTransformer(transformers=[
    ("num",numerical_pipeline,numerical_cols),
    ("cat",categorical_pipeline,categorical_cols)
])

In [ ]:
model_KMeans=Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("model",KMeans())
])

In [ ]:
model_KMeans

In [ ]:
results=[]
for k in range(2,11):
    model_KMeans.set_params(
        model__n_clusters=k,
        model__random_state=42
    )
    labels=model_KMeans.fit_predict(data)
    inertia=model_KMeans.named_steps["model"].inertia_
    x_scaled=preprocessor.fit_transform(data)
    silhouette_score_=silhouette_score(
        x_scaled,
        labels
    )
    results.append({
        "k":k,
        "inertia":inertia,
        "silhouette_score":silhouette_score_
    })

In [ ]:
results=pd.DataFrame(results)

In [ ]:
results

In [ ]:
sns.lineplot(
    x=results["k"],
    y=results["inertia"]
)
plt.show()

In [ ]:
model_KMeans=Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("model",KMeans(n_clusters=4,
    random_state=34))
])

In [ ]:
data["clusters"]=model_KMeans.fit_predict(data)

In [ ]:
plt.scatter(
    data["Age"],
    data["Spending Score (1-100)"],
    c=data["clusters"]
)
plt.title("clusters scatters")
plt.xlabel("age")
plt.ylabel("spending score")
plt.show()